#### Datenvorbereitung für den Ames Housing Price Data Set und Ausgabe von CSV für folgende Modellerstellungen

###### Zweck
Hier werden zentral verschiedene CSV Dateien geschrieben, die in Skripten zur Modellerstellung eingelesen werden.
Explorative Datenanalyse wird in den Notebooks Ames EDA interaktiv.ipynb und Ames EDA handcrafted.ipynb gemacht.

###### Versionsgeschichte
- 1.0 21.08.2023 Willi Hahn Initialversion 
- 1.1 21.09.2025 Willi Hahn Reduziert auf Quantifizierung von Variablen und Ausgabe von CSV.
- 1.2 06.10.2025 Willi Hahn Neue transformierte Variablen und Definition von Dimensionen
- 1.2.1 15.10.2025 Willi Hahn Ames_Num.csv ohne Ausreißer, missing values imputiert, mit allen Variablen numerisch umgewandelt. SalePriceLog, Garage area2, Overall Qual2, Overall Cond2. HouseAge, ReModAge, Neighborhood label encoded
- 1.2.2 25.10.2025 Willi Hahn Ames_NumCat.csv ohne Ausreißer, missing values imputiert. Int64 (erlaubt NaN, ist Pandas extension) umgenaut als int.
- 1.2.1 15.10.2025 Willi Hahn Abgleich der Variablen je Datei. Anpassung pandas 3.


TODO
Neue Variablen:
- RecentRemodel — Binary variable that takes 1 if YearRemodAdd >= YrSold-1
- SqYearBuilt — Continuous variable constructed by exponentiating YearBuilt to the second power
- YearSoldYearBuilt — Binary variable that takes 1 if YearBuilt = YrSold
- AgeAtSale — Continuous variable constructed by subtracting YrBuilt from YrSold
- X['AgeCond'] = X.HouseAge * X.OverallCond
- RR_Proximity — Binary variable that takes 1 if Condition1 ∈ { ‘RRNn’, ‘RRAn’, ‘RRNe’, ‘RRAe’)
- $Saleprice/sqft$/sqft Anomalie!

TODO Modellspezifische Datensätze
- Tree vs. FNN

In [1]:
# notwendige Bibliotheken importieren und konfigurieren
import pandas as pd
_ = pd.set_option('display.max_columns', None) # damit mehr als 20 Spalten angezeigt werden.
_ = pd.set_option('display.min_rows', 8) # damit nicht nur 10 Zeilen mit  ... dazwischen ausgegeben werden
_ = pd.set_option('display.max_rows', 500) # damit nicht nur 10 Zeilen mit  ... dazwischen ausgegeben werden
import numpy as np
import math

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive gemounted!")
except ImportError:
    IS_COLAB = False


In [2]:
# Hilfsfunktionen



def add_dummies(df, columns, drop_first=False, prefix=''):
    """
    Fügt Dummy-Variablen für eine Liste kategorialer Spalten hinzu, idempotent.
    und erhält die Originalvariable.
    Parameter:
    - df: pandas DataFrame
    - columns: Liste von Spaltennamen (strings), die in Dummy-Variablen umgewandelt werden sollen
    - prefix: Präfix für die Dummy-Spaltennamen (string); wenn leer, wird kein '_' verwendet
    - drop_first: Bool, ob die erste Kategorie entfernt werden soll
    
    Rückgabe:
    - df: DataFrame mit Originalspalten und neuen Dummy-Spalten
    """
    df_copy = df.copy()
    
    for col in columns:
        categories = df_copy[col].dropna().unique()
        if drop_first:
            categories = categories[1:]
        
        # Dummy-Spaltennamen erzeugen
        if prefix:
            expected_cols = [f"{prefix}_{col}_{cat}" for cat in categories]
            dummies = pd.get_dummies(df_copy[col], prefix=f"{prefix}_{col}", drop_first=drop_first)
        else:
            expected_cols = [f"{col}_{cat}" for cat in categories]
            dummies = pd.get_dummies(df_copy[col], prefix=col, drop_first=drop_first)
        
        # Nur hinzufügen, wenn Dummy-Spalten noch nicht vorhanden sind
        if not all(col_name in df_copy.columns for col_name in expected_cols):
            df_copy = pd.concat([df_copy, dummies], axis=1)
    
    return df_copy



def print_memory_usage(df, unit='MB', round_to=2):
    """
    Gibt den Gesamtspeicherverbrauch eines DataFrames aus.
    
    Parameter:
    - df: pandas DataFrame
    - unit: 'KB' oder 'MB' (Standard: 'MB')
    - round_to: Anzahl der Nachkommastellen (Standard: 2)
    
    Rückgabe:
    - Speicherverbrauch als float
    """
    factor = 1024 if unit == 'KB' else 1024**2
    usage = df.memory_usage(deep=True).sum() / factor
    print(f"Gesamtspeicherverbrauch: {usage:.{round_to}f} {unit}")
    return usage

In [3]:
# Daten einlesen 


if IS_COLAB:
    path = '/content/drive/MyDrive/Colab Notebooks/AmesHousing.csv'
else:
    path = 'AmesHousing.csv'
#path = 'https://raw.githubusercontent.com/WilliHahn/FHDW/main/AmesHousing.csv'

df = pd.read_csv(path, sep=',') 
df_org = df.copy() # Eine Datenkopie als Referenz behalten
df_num=df.copy() # df_num vorbereiten zum Aufbau numerischer Variablen 
df_numcat=df.copy() #vorbereiten zum Aufbau für Entscheidungsbäume
#df.columns.tolist()


In [4]:
# Definition globaler Konstanten

ONE_HOT_DROP_FIRST=False    # ACHTUNG: Unterschiedliche Modelle brauchen unterschiedliche Behandlung!
                            # Hier wird das Entfernen einer linear abhängigen Variablen jedoch manuell gemacht.

In [5]:
# Data Frame ansehen mit klassischen Methoden aus pandas

#df_num.info(verbose=True)
df_num.info(verbose=True, show_counts=True)
###df.info(verbose=True, show_counts=True) # Namen, Datentypen und Datenanzahl not null
###print (df.describe()) # Lage-, Streuungsparameter je Variable
###df.head(20) # einfache Tabbelle mit scroll Funktion anzeichen


<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style      2930 non

In [6]:
# Ausreißer entfernen
df_num = df_num[df_num['Gr Liv Area'] < 4000] # siehe Originaldoku und Ames Outlier.ipynb

# Datenfehler beheben
df_num.loc[df_num['Order'] ==2261, 'Garage Yr Blt'] = 2007 # Datenfehler, da zuvor 2207
df_num['Sale Type'] = df_num['Sale Type'].str.strip()   # Fehler mit hängendem Leerzeichen entfernen

df_numcat=df_num.copy()

In [7]:
# Variable SalesPrice und Variablen bezogen auf den Verkauf
df_num['SalesPriceLog']=np.log(df_num['SalePrice'] ) # logarithmus naturalis


# Anormale und Familienverkäufe entfernen
#df_num = df_num.loc[~((df_num['Sale Condition'] == 'Abnorml') | (df_num['Sale Condition'] == 'Family') )]
df_num = add_dummies(df_num, columns=['Sale Condition','Sale Type'], drop_first=ONE_HOT_DROP_FIRST)

In [8]:
# Variablen bezogen auf das Gründstück

df_num['Lot Shape'] = df_num['Lot Shape'].map({'Reg':5, 'IR1':2,'IR2':1,'IR3':0}).astype('Int64')
df_num['Land Slope'] = df_num['Land Slope'].map({'Gtl':5,'Mod':2,'Sev':0}).astype('Int64')
df_num['Land Contour'] = df_num['Land Contour'].map({ 'Lvl':5,'Bnk':2,'Low':2,'HLS':1}).astype('Int64') 
df_num['Lot Config'] = df_num['Lot Config'].map({'CulDSac':4, 'Inside':3,'Corner':2,'FR2':1,'FR3':1}).astype('Int64')

# Imputation der Lot Frontage / Gebäudefront mit dem Mittelwert der Nachbarschaft
df_num["Lot Frontage"] = pd.to_numeric(df_num["Lot Frontage"], errors='coerce', downcast="float")
neighborhood_means = df_num.groupby('Neighborhood')['Lot Frontage'].mean()
# fehlende Gebäudefrontmittelwerte mit der overall Mittelwert füllen
neighborhood_means = neighborhood_means.fillna(np.mean(neighborhood_means)) 
df_num['Lot Frontage'] = df_num['Lot Frontage'].fillna(df_num['Neighborhood']) 
# Vorortnamen gegen Gebäudefrontmittelwerte tauschen
df_num['Lot Frontage'] = df_num['Lot Frontage'].map(lambda x: neighborhood_means[x] if type(x)==str else x)
df_numcat['Lot Frontage'] = df_num['Lot Frontage'] 

# Berechne den Median pro Neighborhood für neue Variable NeighborhoodClass
median_prices = df_num.groupby('Neighborhood')['SalePrice'].median()
neighborhood_class = median_prices.apply(    lambda x: 1 if x < 145000 else (2 if x < 290000 else 3))
aggregated = neighborhood_class.rename('NeighborhoodClass').to_frame()
df_num['NeighborhoodClass'] = df_num['Neighborhood'].map(neighborhood_class)
df_num = add_dummies(df_num, columns=['Neighborhood'], drop_first=ONE_HOT_DROP_FIRST)

# 1) Neighbor_int: Ersetze Vorortnamen durch fortlaufende Integerzahl (0,1,2,...)
codes, unique = pd.factorize(df_num['Neighborhood'])
df_num['Neighbor_int'] = codes  # Wertebereich 0 .. (Anzahl Vororte-1)

# 2) Neighbor_intsorted: Fortlaufende Integerzahl, sortiert nach mittlerem SalePrice je Vorort
# Berechne den mittleren Verkaufspreis pro Nachbarschaft
mean_price_per_neigh = df_num.groupby('Neighborhood')['SalePrice'].mean().sort_values()
# Erstelle Mapping: Nachbarschaft -> Rang (beginnend bei 1 für den günstigsten Vorort)
rank_mapping = {neigh: idx+1 for idx, neigh in enumerate(mean_price_per_neigh.index)}
df_num['Neighbor_intsorted'] = df_num['Neighborhood'].map(rank_mapping)


df_num['MS Zoning'] = df_num['MS Zoning'].map({'RP':7,'RL':6,'RM':5,'RH':4,'FV':7,'C (all)':2,'A (agr)':1,'I (all)':2}).astype('int')


df_num.loc[df_num['Condition 1'] == 'RRAn', 'Railroad_Adjacent'] = 1
df_num.loc[df_num['Condition 2'] == 'RRAn', 'Railroad_Adjacent'] = 1
df_num.loc[df_num['Condition 1'] == 'RRAe', 'Railroad_Adjacent'] = 1
df_num.loc[df_num['Condition 2'] == 'RRAe', 'Railroad_Adjacent'] = 1
df_num['Railroad_Adjacent'] = df_num['Railroad_Adjacent'].fillna(0).astype('int')
df_num.loc[df_num['Condition 1'] == 'RRNn', 'Railroad_Near'] = 1
df_num.loc[df_num['Condition 2'] == 'RRNn', 'Railroad_Near'] = 1
df_num.loc[df_num['Condition 1'] == 'RRNe', 'Railroad_Near'] = 1
df_num.loc[df_num['Condition 2'] == 'RRNe', 'Railroad_Near'] = 1
df_num['Railroad_Near'] = df_num['Railroad_Near'].fillna(0).astype('int')
df_num.loc[df_num['Condition 1'] == 'Feedr', 'Street_Feeder'] = 1
df_num.loc[df_num['Condition 2'] == 'Feedr', 'Street_Feeder'] = 1
df_num['Street_Feeder'] = df_num['Street_Feeder'].fillna(0).astype('int')
df_num.loc[df_num['Condition 1'] == 'Artery', 'Street_Aterial'] = 1
df_num.loc[df_num['Condition 2'] == 'Artery', 'Street_Aterial'] = 1
df_num['Street_Aterial'] = df_num['Street_Aterial'].fillna(0).astype('int')
df_num.loc[df_num['Condition 1'] == 'PosA', 'PosFeature_Adjacent'] = 1
df_num.loc[df_num['Condition 2'] == 'PosA', 'PosFeature_Adjacent'] = 1
df_num['PosFeature_Adjacent'] = df_num['PosFeature_Adjacent'].fillna(0).astype('int')
df_num.loc[df_num['Condition 1'] == 'PosN', 'PosFeature_Near'] = 1
df_num.loc[df_num['Condition 2'] == 'PosN', 'PosFeature_Near'] = 1
df_num['PosFeature_Near'] = df_num['PosFeature_Near'].fillna(0).astype('int')

df_num['Paved Drive'] = df_num['Paved Drive'].map({'N':0,'P':1,'Y':2}).astype('int') # None, Partial, Yes
df_num = add_dummies(df_num, columns=['Street'], drop_first=ONE_HOT_DROP_FIRST)


df_num['Alley'] = df_num['Alley'].fillna('NA')
df_num['Alley'] = df_num['Alley'].map({'Pave':3, 'Grvl':2, 'NA':0}).astype('int') # NA als 3 wegen mittlerem Saleprice bei 180k
df_numcat['Alley'] = df_num['Alley'] 





In [9]:
## Variablen bezogen auf Ausstattung des Hauses

df_num['Gr Liv Area2'] = df_num['Gr Liv Area'].apply(lambda x: x**2)
df_num['Gr Liv Area Log'] = df_num['Gr Liv Area'].apply(lambda x: np.log(x))


df_num['Area1st2nd'] = df_num['1st Flr SF'] + df_num['2nd Flr SF']

df_num['Overall Qual2'] = df_num['Overall Qual'].apply(lambda x: x**2)
df_num['Overall Cond2'] = df_num['Overall Cond'].apply(lambda x: x**2)
df_num['Misc Feature'] = df_num['Misc Feature'].fillna('NA')
df_num = add_dummies(df_num, columns=['Misc Feature'], drop_first=ONE_HOT_DROP_FIRST)
df_num['Utilities'] = df_num['Utilities'].fillna('XXX')
df_num['Utilities'] = df_num['Utilities'].map({'AllPub':3,'NoSewr':1,'NoSeWa':1,'ELO':0,'XXX':0}).fillna(0).astype('int')
for i in ['Exter Qual','Exter Cond','Kitchen Qual']:
    df_num[i] = df_num[i].map({'Ex':4,'Gd':3,'TA':2,'Fa':1,'Po':0}).fillna(0).astype('int')
df_num = add_dummies(df_num, columns=['Heating'], drop_first=ONE_HOT_DROP_FIRST)
df_num.drop(['Heating'], axis=1, inplace=True)
df_num['Heating QC'] = df_num['Heating QC'].map({'Ex':4,'Gd':3,'TA':2,'Fa':1,'Po':0}).fillna(0).astype('int')
df_num['Central Air'] = df_num['Central Air'].map({'Y':1,'N':0})
df_num['Electrical'] = df_num['Electrical'].fillna('None')
df_num['Electrical'] = df_num['Electrical'].map({'SBrkr':4,'FuseA':3,'FuseF':2, 'FuseP':1, 'Mix':0, 'None':0}).fillna(0).astype('int')
df_num['Fireplace Qu'] = df_num['Fireplace Qu'].fillna('None')
df_num['Fireplace Qu'] = df_num['Fireplace Qu'].map({'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0,'NA':0}).fillna(0).astype('int')
df_num['Pool QC'] = df_num['Pool QC'].fillna('None')
df_num['Pool QC'] = df_num['Pool QC'].map({'Ex':4,'Gd':3,'TA':2,'Fa':1,'None':0,'NA':0}).fillna(0).astype('int')
df_num['Fence'] = df_num['Fence'].fillna('None')
df_num['Fence'] = df_num['Fence'].map({'GdPrv':4,'GdWo':3,'MnPrv':2,'MnWw':1,'None':0,'NA':0}).fillna(0).astype('int')

df_num.loc[df_num['Exterior 1st'] == 'AsbShng', 'Exterior_AsbShng'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'AsbShng', 'Exterior_AsbShng'] = 1
df_num['Exterior_AsbShng'] = df_num['Exterior_AsbShng'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'BrkComm', 'Exterior_BrkComm'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'BrkComm', 'Exterior_BrkComm'] = 1
df_num['Exterior_BrkComm'] = df_num['Exterior_BrkComm'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'BrkFace', 'Exterior_BrkFace'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'BrkFace', 'Exterior_BrkFace'] = 1
df_num['Exterior_BrkFace'] = df_num['Exterior_BrkFace'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'CBlock', 'Exterior_CBlock'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'CBlock', 'Exterior_CBlock'] = 1
df_num['Exterior_CBlock'] = df_num['Exterior_CBlock'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'CemntBd', 'Exterior_CemntBd'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'CemntBd', 'Exterior_CemntBd'] = 1
df_num['Exterior_CemntBd'] = df_num['Exterior_CemntBd'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'HdBoard', 'Exterior_HdBoard'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'HdBoard', 'Exterior_HdBoard'] = 1
df_num['Exterior_HdBoard'] = df_num['Exterior_HdBoard'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'ImStucc', 'Exterior_ImStucc'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'ImStucc', 'Exterior_ImStucc'] = 1
df_num['Exterior_ImStucc'] = df_num['Exterior_ImStucc'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'MetalSd', 'Exterior_MetalSd'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'MetalSd', 'Exterior_MetalSd'] = 1
df_num['Exterior_MetalSd'] = df_num['Exterior_MetalSd'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'Plywood', 'Exterior_Plywood'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'Plywood', 'Exterior_Plywood'] = 1
df_num['Exterior_Plywood'] = df_num['Exterior_Plywood'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'PreCast', 'Exterior_PreCast'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'PreCast', 'Exterior_PreCast'] = 1
df_num['Exterior_PreCast'] = df_num['Exterior_PreCast'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'Stone', 'Exterior_Stone'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'Stone', 'Exterior_Stone'] = 1
df_num['Exterior_Stone'] = df_num['Exterior_Stone'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'Stucco', 'Exterior_Stucco'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'Stucco', 'Exterior_Stucco'] = 1
df_num['Exterior_Stucco'] = df_num['Exterior_Stucco'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'VinylSd', 'Exterior_VinylSd'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'VinylSd', 'Exterior_VinylSd'] = 1
df_num['Exterior_VinylSd'] = df_num['Exterior_VinylSd'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'Wd Sdng', 'Exterior_Wd Sdng'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'Wd Sdng', 'Exterior_Wd Sdng'] = 1
df_num['Exterior_Wd Sdng'] = df_num['Exterior_Wd Sdng'].fillna(0)
df_num.loc[df_num['Exterior 1st'] == 'WdShing', 'Exterior_WdShing'] = 1
df_num.loc[df_num['Exterior 2nd'] == 'WdShing', 'Exterior_WdShing'] = 1
df_num['Exterior_WdShing'] = df_num['Exterior_WdShing'].fillna(0)


df_num['Mas Vnr Type'] = df_num['Mas Vnr Type'].fillna('None')
#df_num = add_dummies(df_num, columns=['Mas Vnr Type'], drop_first=ONE_HOT_DROP_FIRST)
df_num["Mas Vnr Area"] = pd.to_numeric(df_num["Mas Vnr Area"], errors='coerce', downcast="float")
df_num['Mas Vnr Area'] = df_num['Mas Vnr Area'].fillna(0)


df_numcat['Mas Vnr Type'] = df_numcat['Mas Vnr Type'].fillna('None') 
df_numcat['Mas Vnr Area'] = df_numcat['Mas Vnr Area'].fillna(0)
df_numcat['Misc Feature'] = df_numcat['Misc Feature'].fillna('None')
df_numcat['Pool QC'] = df_numcat['Pool QC'].fillna('None')
df_numcat['Fireplace Qu'] = df_numcat['Fireplace Qu'].fillna(0)
df_numcat['Fence'] = df_numcat['Fence'].fillna('None')
df_numcat['Electrical'] = df_numcat['Electrical'].fillna('None')




In [10]:
## Variablen bezogen auf das Haus
df_num = add_dummies(df_num, columns=['House Style', 'Bldg Type', 'Foundation'], drop_first=ONE_HOT_DROP_FIRST)
df_num['HouseAge'] = df_num['Yr Sold'] - df_num['Year Built'].astype('int')
df_num['HouseRemodelAge'] = df_num['Yr Sold'] - df_num['Year Remod/Add'].astype('int')
df_num['HouseIsreModeled'] = np.where(df_num['Year Remod/Add']  == df_num['Year Built'], 0, 1)
df_num['HouseIsNew'] = np.where(df_num['Yr Sold'] == df_num['Year Built'], 1, 0)
# Bedingungen für RenovationNeeded definieren
CONDITIONS = {
    'Bsmt Cond': lambda x: x.isin(['Po', 'Fa']),
    'Electrical': lambda x: x == 'FuseP',
    'Exter Qual': lambda x: x.isin(['Po', 'Fa']),
    'Exterior 1st': lambda x: x == 'AsbShng',
    'Fireplace Qu': lambda x: x.isin(['Po', 'Fa']),
    'Functional': lambda x: x.isin(['Sal', 'Sev', 'Maj2']),
    'Garage Cond': lambda x: x.isin(['Po', 'Fa']),
    'Garage Qual': lambda x: x.isin(['Po', 'Fa']),
    'Heating QC': lambda x: x.isin(['Po', 'Fa']),
    'Kitchen Qual': lambda x: x.isin(['Po', 'Fa']),
    'Low Qual Fin SF': lambda x: pd.to_numeric(x, errors='coerce') > 100,
    'Overall Cond': lambda x: pd.to_numeric(x, errors='coerce').isin([1, 2]),
    'Overall Qual': lambda x: pd.to_numeric(x, errors='coerce').isin([0, 1, 2]),
    'Paved Drive': lambda x: x == 'N',
    'Pool QC': lambda x: x == 'Fa',
    'Utilities': lambda x: x.isin(['NoSewr', 'NoSeWa', 'ELO'])
}

# Erstelle die RenovationNeeded-Spalte
mask = pd.Series(False, index=df.index)

for col, condition_func in CONDITIONS.items():
    if col in df.columns:
        try:
            condition_mask = condition_func(df[col].fillna(''))
            mask = mask | condition_mask
        except Exception as e:
            print(f"Fehler bei Spalte {col}: {e}")
            continue

df_num['RenovationNeeded'] = mask.astype(int)
df_num['Bsmt Full Bath'] = df_num['Bsmt Full Bath'].fillna(0)
df_num['Bsmt Half Bath'] = df_num['Bsmt Half Bath'].fillna(0)
df_num["Bsmt Full Bath"] = pd.to_numeric(df_num["Bsmt Full Bath"], errors='coerce', downcast="float")
df_num["Bsmt Half Bath"] = pd.to_numeric(df_num["Bsmt Half Bath"], errors='coerce', downcast="float")
df_num['BathroomsTotal'] = df_num['Full Bath'] + df_num['Bsmt Full Bath'] + 0.5 * df_num['Bsmt Half Bath'] + 0.5 * df_num['Half Bath'] 
df_num['BathroomsTotal'] = df_num['BathroomsTotal'].fillna(0)
df_num['Functional'] = df_num['Functional'].map({'Typ':7,'Min1':6,'Min2':5,'Mod':4,'Maj1':3,'Maj2':2,'Sev':1,'Sal':0})
df_num['Functional_bin'] = df_num['Functional'].apply(lambda x: 0 if x <= 2 else 1)
df_num['PorchSFTotal'] = ( df_num['Wood Deck SF'] + df_num['Open Porch SF'] + df_num['Enclosed Porch'] +
    df_num['3Ssn Porch'] + df_num['Screen Porch'] )


# Abbruchhäuser entfernen
#df_num = df_num.loc[~((df_num['Functional'] == 'Sal') | (df_num['Functional'] == 'Sev'))]  



In [11]:
df_num.info(verbose=True)

<class 'pandas.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 195 columns):
 #    Column                  Dtype  
---   ------                  -----  
 0    Order                   int64  
 1    PID                     int64  
 2    MS SubClass             int64  
 3    MS Zoning               int64  
 4    Lot Frontage            float64
 5    Lot Area                int64  
 6    Street                  str    
 7    Alley                   int64  
 8    Lot Shape               Int64  
 9    Land Contour            Int64  
 10   Utilities               int64  
 11   Lot Config              Int64  
 12   Land Slope              Int64  
 13   Neighborhood            str    
 14   Condition 1             str    
 15   Condition 2             str    
 16   Bldg Type               str    
 17   House Style             str    
 18   Overall Qual            int64  
 19   Overall Cond            int64  
 20   Year Built              int64  
 21   Year Remod/Add          int6

In [12]:
## Variablen bezogen auf Dach und Dachausbau
df_num = add_dummies(df_num, columns=['Roof Matl', 'Roof Style'], drop_first=ONE_HOT_DROP_FIRST)


In [13]:
## Variablen bezogen auf den Keller
df_num["BsmtFin SF 1"] = pd.to_numeric(df_num["BsmtFin SF 1"], errors='coerce', downcast="float")
df_num["BsmtFin SF 2"] = pd.to_numeric(df_num["BsmtFin SF 2"], errors='coerce', downcast="float")
df_num['BsmtFin SF 1'] = df_num['BsmtFin SF 1'].fillna(0)
df_num['BsmtFin SF 2'] = df_num['BsmtFin SF 2'].fillna(0)
df_num['Bsmt Unf SF'] = df_num['Bsmt Unf SF'].fillna(0)
df_num['Total Bsmt SF'] = df_num['BsmtFin SF 1'] + df_num['BsmtFin SF 2']
df_num['Bsmt Half Bath'] = df_num['Bsmt Full Bath'].fillna(0)
df_num['Bsmt Full Bath'] = df_num['Bsmt Full Bath'].fillna(0)
df_num['Bsmt Qual'] = df_num['Bsmt Qual'].fillna('None')
df_num['Bsmt Qual'] = df_num['Bsmt Qual'].map({'Ex':5,'Gd':5,'TA':4,'Fa':3,'Po':2,'None':0}).astype('int')
#df_num['Bsmt Qual'] = df_num['Bsmt Qual'].map({'Ex':24,'Gd':12,'TA':6,'Fa':3,'Po':1,'None':0}).astype('Int64')
df_num['Bsmt Cond'] = df_num['Bsmt Cond'].fillna('None')
df_num['Bsmt Cond'] = df_num['Bsmt Cond'].map({'Ex':5,'Gd':5,'TA':4,'Fa':3,'Po':2,'None':0}).astype('int')
#df_num['Bsmt Cond'] = df_num['Bsmt Cond'].map({'Ex':24,'Gd':12,'TA':6,'Fa':3,'Po':1,'None':0}).astype('Int64')
df_num['BsmtFin Type 1'] = df_num['BsmtFin Type 1'].fillna('None')
df_num['BsmtFin Type 1'] = df_num['BsmtFin Type 1'].map({'GLQ':5,'ALQ':4,'BLQ':3,'Rec':2,'LwQ':2,'Unf':1,'None':0}).astype('int')
df_num['BsmtFin Type 2'] = df_num['BsmtFin Type 2'].fillna('None')
df_num['BsmtFin Type 2'] = df_num['BsmtFin Type 2'].map({'GLQ':5,'ALQ':4,'BLQ':3,'Rec':2,'LwQ':2,'Unf':1,'None':0}).astype('int')
df_num['Bsmt Exposure'] = df_num['Bsmt Exposure'].fillna('None')
df_num['Bsmt Exposure'] = df_num['Bsmt Exposure'].map({'Gd':3,'Av':2,'Mn':1,'No':0,'None':0}).astype('int')


df_numcat['Bsmt Half Bath'] = df_numcat['Bsmt Full Bath'].fillna(0)
df_numcat['Bsmt Full Bath'] = df_numcat['Bsmt Full Bath'].fillna(0)
df_numcat['Bsmt Unf SF'] = df_numcat['Bsmt Unf SF'].fillna(0)
df_numcat['BsmtFin SF 1'] = df_numcat['BsmtFin SF 1'].fillna(0)
df_numcat['BsmtFin SF 2'] = df_numcat['BsmtFin SF 2'].fillna(0)
df_numcat['Total Bsmt SF'] = df_numcat['BsmtFin SF 1'] + df_numcat['BsmtFin SF 2']
df_numcat['Bsmt Qual'] = df_numcat['Bsmt Qual'].fillna('None')
df_numcat['Bsmt Cond'] = df_numcat['Bsmt Cond'].fillna('None')
df_numcat['BsmtFin Type 1'] = df_numcat['BsmtFin Type 1'].fillna('None')
df_numcat['BsmtFin Type 2'] = df_numcat['BsmtFin Type 2'].fillna('None')
df_numcat['Bsmt Exposure'] = df_numcat['Bsmt Exposure'].fillna('None')


In [14]:
## Variablen bezogen auf die Garage
# Garage Cars – einfach 0 für NaN
df_num['Garage Cars'] = df_num['Garage Cars'].fillna(0)

# Garage Type
df_num['Garage Type'] = df_num['Garage Type'].fillna('None')
df_num = add_dummies(df_num, columns=['Garage Type'], drop_first=ONE_HOT_DROP_FIRST)

# Garage Finish
df_num['Garage Finish'] = df_num['Garage Finish'].fillna('None')
df_num.loc[df_num['Garage Finish'] == "", 'Garage Finish'] = 'None'
df_num['Garage Finish'] = df_num['Garage Finish'].map({'Fin':3,'RFn':2,'Unf':1,'None':0,'NA':0,'0':0}).fillna(0).astype(int)

# Garage Qual & Cond (analog)
df_num['Garage Qual'] = df_num['Garage Qual'].fillna('None')
df_num['Garage Qual'] = df_num['Garage Qual'].map({'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0}).fillna(0).astype(int)

df_num['Garage Cond'] = df_num['Garage Cond'].fillna('None')
df_num['Garage Cond'] = df_num['Garage Cond'].map({'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0}).fillna(0).astype(int)

# Garage Yr Blt – robust behandeln
# Zuerst NaN durch Year Built ersetzen, aber nur wenn Year Built selbst nicht NaN ist
df_num['Garage Yr Blt'] = df_num['Garage Yr Blt'].fillna(df_num['Year Built'])
# Leere Strings durch Year Built der gleichen Zeile ersetzen
mask_empty = df_num['Garage Yr Blt'] == ""
df_num.loc[mask_empty, 'Garage Yr Blt'] = df_num.loc[mask_empty, 'Year Built']
# Jetzt in numerisch umwandeln, dabei alle verbleibenden Fehler zu NaN
df_num['Garage Yr Blt'] = pd.to_numeric(df_num['Garage Yr Blt'], errors='coerce')
# Falls jetzt noch NaN da sind (weil Year Built auch NaN war) -> durch 0 ersetzen
df_num['Garage Yr Blt'] = df_num['Garage Yr Blt'].fillna(0).astype(int)

# Garage Area
df_num['Garage Area'] = pd.to_numeric(df_num['Garage Area'], errors='coerce').fillna(0)
df_num['Garage Area2'] = df_num['Garage Area'] ** 2

# Garage Age (Yr Sold muss numerisch und ohne NaN sein)
df_num['Yr Sold'] = pd.to_numeric(df_num['Yr Sold'], errors='coerce').fillna(0).astype(int)
df_num['GarageAge'] = df_num['Yr Sold'] - df_num['Garage Yr Blt']
df_num['GarageAge'] = df_num['GarageAge'].fillna(0).astype(int)

df_numcat['Garage Yr Blt'] = df_num['Garage Yr Blt']
df_numcat['Garage Qual'] = df_num['Garage Qual']
df_numcat['Garage Cond'] = df_num['Garage Cond']
df_numcat['Garage Type'] = df_num['Garage Type']
df_numcat['Garage Area'] = df_num['Garage Area']
df_numcat['Garage Cars'] = df_num['Garage Cars']
df_numcat['Garage Finish'] = df_num['Garage Finish']


In [15]:
# Variablen entfernen

# Spalte Order bleibt, weil  als eindeutiger Schlüssel verwendet. besonders zur Anzeige in interaktiven plot. Order muss bei Modellbildung entfernt werden.
df_num.drop('PID',axis=1,inplace=True) 
df_num.drop(['Condition 1', 'Condition 2'],axis=1,inplace=True) # 
df_num.drop(['Exterior 1st', 'Exterior 2nd'],axis=1,inplace=True) # 
#df_num.drop(['Functional'],axis=1,inplace=True) 

df_num.drop(['Mo Sold', 'Yr Sold'],axis=1,inplace=True) # Data Leakage, wird für Datenfehlererkennung benutzt.

df_num.drop(['Neighborhood', 'Bldg Type', 'House Style', 'Street', 'Roof Style', 'Roof Matl', 'Mas Vnr Type', 'Foundation', 'Garage Type', 
            'Misc Feature', 'Sale Type', 'Sale Condition'],axis=1,inplace=True) # one hot encode Originalvariablen


In [16]:
# letzter Check

#display (df_numcat[df_numcat['Mas Vnr Type'].isna() | df_numcat['Mas Vnr Type'].str.strip().eq('') | df_numcat['Mas Vnr Type'].isnull()])

def analyze_nulls(df):
    # Fehlende Werte analysieren
    null_stats = pd.DataFrame(df.isna().sum(), columns=['missing_value_count'])
    null_stats['% of dataset'] = np.round(null_stats['missing_value_count'] / df.shape[0] * 100, 2)
    missing_summary = null_stats[null_stats['missing_value_count'] > 0].sort_values(by='missing_value_count', ascending=False)
    return missing_summary


print (analyze_nulls (df_num))
print (analyze_nulls (df_numcat))




Empty DataFrame
Columns: [missing_value_count, % of dataset]
Index: []
Empty DataFrame
Columns: [missing_value_count, % of dataset]
Index: []


In [17]:
df_num.info(verbose=True)

<class 'pandas.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 198 columns):
 #    Column                  Dtype  
---   ------                  -----  
 0    Order                   int64  
 1    MS SubClass             int64  
 2    MS Zoning               int64  
 3    Lot Frontage            float64
 4    Lot Area                int64  
 5    Alley                   int64  
 6    Lot Shape               Int64  
 7    Land Contour            Int64  
 8    Utilities               int64  
 9    Lot Config              Int64  
 10   Land Slope              Int64  
 11   Overall Qual            int64  
 12   Overall Cond            int64  
 13   Year Built              int64  
 14   Year Remod/Add          int64  
 15   Mas Vnr Area            float32
 16   Exter Qual              int64  
 17   Exter Cond              int64  
 18   Bsmt Qual               int64  
 19   Bsmt Cond               int64  
 20   Bsmt Exposure           int64  
 21   BsmtFin Type 1          int6

In [18]:
# Hilfsausgabe, um die vorhandenen Variablen durch Copy&paste bei .to_csv einzufügen.

# Numerische Spalten extrahieren
sorted_columns = sorted(df_num.select_dtypes(include='number').columns.tolist())

# alle Variablen
sorted_columns = sorted(df_num.columns.tolist())


# Spalten in Gruppen zu je 6 Elementen aufteilen
for i in range(0, len(sorted_columns), 6):
    chunk = sorted_columns[i:i+6]
    quoted_chunk = [f"'{col}'" for col in chunk]
    print(', '.join(quoted_chunk)+ ',')

'1st Flr SF', '2nd Flr SF', '3Ssn Porch', 'Alley', 'Area1st2nd', 'BathroomsTotal',
'Bedroom AbvGr', 'Bldg Type_1Fam', 'Bldg Type_2fmCon', 'Bldg Type_Duplex', 'Bldg Type_Twnhs', 'Bldg Type_TwnhsE',
'Bsmt Cond', 'Bsmt Exposure', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Bsmt Qual', 'Bsmt Unf SF',
'BsmtFin SF 1', 'BsmtFin SF 2', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Central Air', 'Electrical',
'Enclosed Porch', 'Exter Cond', 'Exter Qual', 'Exterior_AsbShng', 'Exterior_BrkComm', 'Exterior_BrkFace',
'Exterior_CBlock', 'Exterior_CemntBd', 'Exterior_HdBoard', 'Exterior_ImStucc', 'Exterior_MetalSd', 'Exterior_Plywood',
'Exterior_PreCast', 'Exterior_Stone', 'Exterior_Stucco', 'Exterior_VinylSd', 'Exterior_Wd Sdng', 'Exterior_WdShing',
'Fence', 'Fireplace Qu', 'Fireplaces', 'Foundation_BrkTil', 'Foundation_CBlock', 'Foundation_PConc',
'Foundation_Slab', 'Foundation_Stone', 'Foundation_Wood', 'Full Bath', 'Functional', 'Functional_bin',
'Garage Area', 'Garage Area2', 'Garage Cars', 'Garage Cond', 'Gara

In [19]:
# Ausgabe von CSV für die Modellbildungen und weitere Analysen
# Folgende Analyse in Ames Regression interaktiv.ipynb und alle Variablen einlesen und die Varianten dann per Auswahl selektieren


#df_num = df_num.copy()
#df_num[df_num.select_dtypes(include='bool').columns] = df_num.select_dtypes(include='bool').astype(int)

# minimale Variablenanzahl, 16 unabhängige Variablen
df_num[ ['Order', 'Lot Area', 'Gr Liv Area', 'Overall Qual', 'Exter Qual', 'Kitchen Qual', 'Garage Area', 'PorchSFTotal', 'BathroomsTotal'
        , 'HouseAge', 'HouseRemodelAge', 'Total Bsmt SF', 'Railroad_Near', 'Railroad_Adjacent', 'NeighborhoodClass', 'Sale Condition_AdjLand'
        , 'Sale Condition_Alloca', 'Sale Condition_Normal', 'Sale Condition_Partial', 'Sale Type_COD', 'Sale Type_CWD', 'Sale Type_Con', 'Sale Type_ConLD'
        , 'Sale Type_ConLI', 'Sale Type_ConLw', 'Sale Type_New', 'Sale Type_Oth', 'Sale Type_VWD', 'Sale Type_WD', 
         'SalePrice']].to_csv('Ames_Num_Small_NoDropFirst.csv', index=False)
df_num[ ['Order', 'Lot Area', 'Gr Liv Area', 'Overall Qual', 'Exter Qual', 'Kitchen Qual', 'Garage Area', 'PorchSFTotal', 'BathroomsTotal'
        , 'HouseAge', 'HouseRemodelAge', 'Total Bsmt SF', 'Railroad_Near', 'Railroad_Adjacent', 'NeighborhoodClass'  #, 'Sale Condition_AdjLand' betrifft nur 12 Häuser
        , 'Sale Condition_Alloca', 'Sale Condition_Normal', 'Sale Condition_Partial', 'Sale Type_COD', 'Sale Type_CWD', 'Sale Type_ConLD'
        , 'Sale Type_ConLI', 'Sale Type_ConLw', 'Sale Type_New', 'Sale Type_Oth', 'Sale Type_VWD', 'Sale Type_WD', # 'Sale Type_Con' betrifft nur 5 Häuser
         'SalePrice']].to_csv('Ames_Num_Small_DropFirst.csv', index=False)

# folgende mit quadrierten Variablen zu Analyse der Auswirkungen bei Modellbildung
df_num[ ['Order', 'Lot Area', 'Gr Liv Area', 'Overall Qual', 'Overall Qual2', 'Exter Qual', 'Kitchen Qual', 'Garage Area', 'Garage Area2', 'PorchSFTotal', 'BathroomsTotal'
        , 'HouseAge', 'HouseRemodelAge', 'Total Bsmt SF', 'Railroad_Near', 'Railroad_Adjacent', 'NeighborhoodClass', 'Sale Condition_AdjLand'
        , 'Sale Condition_Alloca', 'Sale Condition_Normal', 'Sale Condition_Partial', 'Sale Type_COD', 'Sale Type_CWD', 'Sale Type_Con', 'Sale Type_ConLD'
        , 'Sale Type_ConLI', 'Sale Type_ConLw', 'Sale Type_New', 'Sale Type_Oth', 'Sale Type_VWD', 'Sale Type_WD', 
         'SalePrice']].to_csv('Ames_Num_Small2_NoDropFirst.csv', index=False)
df_num[ ['Order', 'Lot Area', 'Gr Liv Area', 'Overall Qual', 'Overall Qual2', 'Exter Qual', 'Kitchen Qual', 'Garage Area', 'Garage Area2', 'PorchSFTotal', 'BathroomsTotal'
        , 'HouseAge', 'HouseRemodelAge', 'Total Bsmt SF', 'Railroad_Near', 'Railroad_Adjacent', 'NeighborhoodClass'  #, 'Sale Condition_AdjLand' betrifft nur 12 Häuser
        , 'Sale Condition_Alloca', 'Sale Condition_Normal', 'Sale Condition_Partial', 'Sale Type_COD', 'Sale Type_CWD', 'Sale Type_ConLD'
        , 'Sale Type_ConLI', 'Sale Type_ConLw', 'Sale Type_New', 'Sale Type_Oth', 'Sale Type_VWD', 'Sale Type_WD', # 'Sale Type_Con' betrifft nur 5 Häuser
         'SalePrice']].to_csv('Ames_Num_Small2_DropFirst.csv', index=False)

df_num[ ['Order', 'Lot Area', 'Gr Liv Area', 'Area1st2nd', 'Overall Qual', 'Exter Qual', 'Kitchen Qual', 'Garage Area', 'PorchSFTotal', 'BathroomsTotal'
        , 'HouseAge', 'HouseRemodelAge', 'Total Bsmt SF', 'Railroad_Near', 'Railroad_Adjacent', 'NeighborhoodClass' , 'Neighbor_intsorted' 
        , 'Central Air'
        , 'Sale Type_New', 'Sale Condition_Partial', 'Sale Type_COD' , 'Sale Type_WD' # Sale Type_* nur die bei RF wichtigen Features
        , 'SalePrice']].to_csv('Ames_Num_Small3.csv', index=False)


# folgende mit quadrieten Vars, SalesPriceLog und no drop bei one hot encoded Vars
df_numcat.to_csv('Ames_NumCat_All.csv', index=False)
df_num.to_csv('Ames_Num_All.csv', index=False)

# Ames_Num_All_clean.csv alle Variablen, ohne drop first, ohne quadrierte Vars
df_num[['1st Flr SF', '2nd Flr SF', '3Ssn Porch', 'Alley', 'BathroomsTotal', 'Bedroom AbvGr',
'Bldg Type_1Fam', 'Bldg Type_2fmCon', 'Bldg Type_Duplex', 'Bldg Type_Twnhs', 'Bldg Type_TwnhsE', 'Bsmt Cond',
'Bsmt Exposure', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Bsmt Qual', 'Bsmt Unf SF', 'BsmtFin SF 1',
'BsmtFin SF 2', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Central Air', 'Electrical', 'Enclosed Porch',
'Exter Cond', 'Exter Qual', 'Exterior_AsbShng', 'Exterior_BrkComm', 'Exterior_BrkFace', 'Exterior_CBlock',
'Exterior_CemntBd', 'Exterior_HdBoard', 'Exterior_ImStucc', 'Exterior_MetalSd', 'Exterior_Plywood', 'Exterior_PreCast',
'Exterior_Stone', 'Exterior_Stucco', 'Exterior_VinylSd', 'Exterior_Wd Sdng', 'Exterior_WdShing', 'Fence',
'Fireplace Qu', 'Fireplaces', 'Foundation_BrkTil', 'Foundation_CBlock', 'Foundation_PConc', 'Foundation_Slab',
'Foundation_Stone', 'Foundation_Wood', 'Full Bath', 'Functional', 'Functional_bin', 'Garage Area',
'Garage Cars', 'Garage Cond', 'Garage Finish', 'Garage Qual', 'Garage Type_2Types',
'Garage Type_Attchd', 'Garage Type_Basment', 'Garage Type_BuiltIn', 'Garage Type_CarPort', 'Garage Type_Detchd', 'Garage Type_None',
'GarageAge', 'Gr Liv Area', 'Half Bath',
'Heating QC', 'Heating_Floor', 'Heating_GasA', 'Heating_GasW', 'Heating_Grav', 'Heating_OthW',
'Heating_Wall', 'House Style_1.5Fin', 'House Style_1.5Unf', 'House Style_1Story', 'House Style_2.5Fin', 'House Style_2.5Unf',
'House Style_2Story', 'House Style_SFoyer', 'House Style_SLvl', 'HouseAge', 'HouseIsNew', 'HouseIsreModeled',
'HouseRemodelAge', 'Kitchen AbvGr', 'Kitchen Qual', 'Land Contour', 'Land Slope', 'Lot Area',
'Lot Config', 'Lot Frontage', 'Lot Shape', 'Low Qual Fin SF', 'MS SubClass', 'MS Zoning',
'Mas Vnr Area', 'Misc Feature_Gar2', 'Misc Feature_NA', 'Misc Feature_Othr', 'Misc Feature_Shed', 'Misc Feature_TenC',
'Misc Val', 'NeighborhoodClass', 'Neighborhood_Blmngtn', 'Neighborhood_Blueste', 'Neighborhood_BrDale', 'Neighborhood_BrkSide',
'Neighborhood_ClearCr', 'Neighborhood_CollgCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards', 'Neighborhood_Gilbert', 'Neighborhood_Greens',
'Neighborhood_GrnHill', 'Neighborhood_IDOTRR', 'Neighborhood_Landmrk', 'Neighborhood_MeadowV', 'Neighborhood_Mitchel', 'Neighborhood_NAmes',
'Neighborhood_NPkVill', 'Neighborhood_NWAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_OldTown', 'Neighborhood_SWISU',
'Neighborhood_Sawyer', 'Neighborhood_SawyerW', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'Neighborhood_Timber', 'Neighborhood_Veenker',
'Open Porch SF', 'Order', 'Overall Cond', 'Overall Qual', 
'Paved Drive', 'Pool Area', 'Pool QC', 'PorchSFTotal', 'PosFeature_Adjacent', 'PosFeature_Near',
'Railroad_Adjacent', 'Railroad_Near', 'RenovationNeeded', 'Roof Matl_CompShg', 'Roof Matl_Membran', 'Roof Matl_Metal',
'Roof Matl_Roll', 'Roof Matl_Tar&Grv', 'Roof Matl_WdShake', 'Roof Matl_WdShngl', 'Roof Style_Flat', 'Roof Style_Gable',
'Roof Style_Gambrel', 'Roof Style_Hip', 'Roof Style_Mansard', 'Roof Style_Shed', 'Sale Condition_Abnorml', 'Sale Condition_AdjLand',
'Sale Condition_Alloca', 'Sale Condition_Family', 'Sale Condition_Normal', 'Sale Condition_Partial', 'Sale Type_COD', 'Sale Type_CWD',
'Sale Type_Con', 'Sale Type_ConLD', 'Sale Type_ConLI', 'Sale Type_ConLw', 'Sale Type_New', 'Sale Type_Oth',
'Sale Type_VWD', 'Sale Type_WD', 'SalePrice', 'Screen Porch', 'Street_Aterial',
'Street_Feeder', 'Street_Grvl', 'Street_Pave', 'TotRms AbvGrd', 'Total Bsmt SF', 'Utilities',
'Wood Deck SF']].to_csv('Ames_Num_All_clean.csv', index=False)


df_num[ ['Order', 'Functional', 'Functional_bin', 'Lot Area', 'Gr Liv Area', 'SalePrice',
'Bsmt Cond', 'Bsmt Qual', 'Exter Cond', 'Exter Qual', 'Kitchen Qual',  'Overall Cond', 'Overall Qual'
]].to_csv('Ames_AbbruchCluster_1.csv', index=False)

df_num[ ['Order', 'Functional', 'Functional_bin', 'Lot Area', 'Gr Liv Area', 'SalePrice', 'Bsmt Cond', 'Bsmt Qual', 'Exter Cond', 'Exter Qual', 'Kitchen Qual',  'Overall Cond', 'Overall Qual',
'Bsmt Unf SF','BsmtFin Type 1', 'BsmtFin Type 2', 'Central Air', 'Electrical',
'Exterior_AsbShng', 'Exterior_BrkComm', 'Exterior_BrkFace','Exterior_CBlock', 'Exterior_CemntBd', 'Exterior_HdBoard', 'Exterior_ImStucc', 'Exterior_MetalSd', 'Exterior_Plywood',
'Exterior_PreCast', 'Exterior_Stone', 'Exterior_Stucco', 'Exterior_VinylSd', 'Exterior_Wd Sdng', 'Exterior_WdShing',
'Fireplace Qu', 'Foundation_BrkTil', 'Foundation_CBlock','Foundation_PConc', 'Foundation_Slab', 'Foundation_Stone', 'Foundation_Wood', 
'Garage Cond', 'Garage Finish', 'Garage Qual', 'Half Bath', 'Heating QC', 'Heating_Floor', 'Heating_GasA', 'Heating_GasW',
'Heating_Grav', 'Heating_OthW', 'Heating_Wall', 'House Style_1.5Unf',
'House Style_2.5Unf', 'HouseIsNew', 'HouseIsreModeled', 'Kitchen Qual', 'Low Qual Fin SF', 'MS SubClass', 'MS Zoning', 
'Paved Drive', 'Pool QC', 'Roof Matl_CompShg', 'Roof Matl_Membran',
'Roof Matl_Metal', 'Roof Matl_Roll', 'Roof Matl_Tar&Grv', 'Roof Matl_WdShake', 'Roof Matl_WdShngl', 
'Roof Style_Flat', 'Roof Style_Gable', 'Roof Style_Gambrel', 'Roof Style_Hip', 'Roof Style_Mansard', 'Roof Style_Shed',
'Street_Grvl', 'Street_Pave', 'Utilities'
]].to_csv('Ames_AbbruchCluster_2.csv', index=False)



In [20]:
# Order mit ähnlichen Merkmale aber großenPreisunterschieden löschen
# Siehe Ames EDA handcrafted.ipynb

def remove_extracted_orders(df_num, extracted_orders_file='Ames_GrossePreisunterschiede.csv', order_column='Order'):
    """
    Liest extrahierte Orders aus einer CSV-Datei ein und entfernt sie aus dem DataFrame.
    
    Parameters:
    -----------
    df_num : pandas.DataFrame
        Der DataFrame, aus dem die Orders entfernt werden sollen.
    extracted_orders_file : str, optional
        Pfad zur CSV-Datei mit den extrahierten Orders (Standard: 'extracted_orders.csv').
    order_column : str, optional
        Name der Order-Spalte im DataFrame (Standard: 'Order').
    
    Returns:
    --------
    pandas.DataFrame
        Der bereinigte DataFrame ohne die extrahierten Orders.
    dict
        Statistiken über die Löschoperation.
    """
    
    # 1. Extrahierten Orders aus CSV-Datei einlesen
    try:
        extracted_df = pd.read_csv(extracted_orders_file)
        
        # Prüfe, ob die Datei die erwartete Spalte hat
        if 'Extracted_Order' in extracted_df.columns:
            orders_to_remove = extracted_df['Extracted_Order'].astype(int).tolist()
        elif 'Order' in extracted_df.columns:
            orders_to_remove = extracted_df['Order'].astype(int).tolist()
        else:
            # Versuche, die erste Spalte als Orders zu interpretieren
            first_column = extracted_df.columns[0]
            orders_to_remove = extracted_df[first_column].astype(int).tolist()
            
        print(f"Eingelesene Orders aus {extracted_orders_file}: {len(orders_to_remove)}")
        
    except FileNotFoundError:
        print(f"FEHLER: Datei {extracted_orders_file} nicht gefunden.")
        print("Stelle sicher, dass die Extraktionsfunktion zuerst ausgeführt wurde.")
        return df_num, {'error': 'Datei nicht gefunden'}
    
    except Exception as e:
        print(f"FEHLER beim Einlesen der Orders: {e}")
        return df_num, {'error': str(e)}
    
    # 2. Prüfe, ob die Order-Spalte im DataFrame existiert
    if order_column not in df_num.columns:
        print(f"FEHLER: Order-Spalte '{order_column}' nicht im DataFrame gefunden.")
        print(f"Verfügbare Spalten: {list(df_num.columns)}")
        return df_num, {'error': f'Order-Spalte {order_column} nicht gefunden'}
    
    # 3. Ermittle die Anzahl der zu entfernenden Datensätze vor der Löschung
    original_count = len(df_num)
    orders_in_df = set(df_num[order_column].astype(int).tolist())
    orders_to_remove_set = set(orders_to_remove)
    
    # Finde Orders, die tatsächlich im DataFrame existieren
    existing_orders_to_remove = list(orders_in_df.intersection(orders_to_remove_set))
    non_existing_orders = list(orders_to_remove_set - orders_in_df)
    
    print(f"\nVor der Löschung:")
    print(f"- Gesamtanzahl Datensätze: {original_count}")
    print(f"- Zu entfernende Orders (aus Datei): {len(orders_to_remove)}")
    print(f"- Davon im DataFrame vorhanden: {len(existing_orders_to_remove)}")
    
    if non_existing_orders:
        print(f"- Nicht im DataFrame gefunden (werden ignoriert): {len(non_existing_orders)}")
        if len(non_existing_orders) <= 10:
            print(f"  Details: {sorted(non_existing_orders)}")
    
    # 4. Filtere den DataFrame: behalte nur Zeilen, deren Order NICHT in der Liste ist
    if existing_orders_to_remove:
        # Erstelle eine Maske: True für Zeilen, die behalten werden sollen
        mask = ~df_num[order_column].astype(int).isin(existing_orders_to_remove)
        
        # Trenne die zu löschenden Zeilen für etwaige spätere Analyse
        rows_to_remove = df_num[~mask].copy()
        df_cleaned = df_num[mask].copy()
        
        # Statistiken
        removed_count = len(rows_to_remove)
        remaining_count = len(df_cleaned)
        
        print(f"\nNach der Löschung:")
        print(f"- Entfernte Datensätze: {removed_count}")
        print(f"- Verbleibende Datensätze: {remaining_count}")
        print(f"- Reduktion: {removed_count/original_count*100:.2f}%")
        
        
        # Statistiken für die Rückgabe
        stats = {
            'original_count': original_count,
            'removed_count': removed_count,
            'remaining_count': remaining_count,
            'orders_removed': existing_orders_to_remove,
            'orders_not_found': non_existing_orders,
            'removed_file': removed_rows_file if 'removed_rows_file' in locals() else None
        }
        
        return df_cleaned, stats
    
    else:
        print("\nKeine der angegebenen Orders wurde im DataFrame gefunden.")
        print("Der DataFrame bleibt unverändert.")
        
        stats = {
            'original_count': original_count,
            'removed_count': 0,
            'remaining_count': original_count,
            'orders_removed': [],
            'orders_not_found': orders_to_remove,
            'removed_file': None
        }
        
        return df_num, stats

data_file='Ames_Num_All_clean.csv'
extracted_orders_file='Ames_GrossePreisunterschiede.csv'

print(f"Verarbeite Daten aus {data_file}")

# 1. Daten laden
df_original = pd.read_csv(data_file)
print(f"Originaler DataFrame: {len(df_original)} Zeilen, {len(df_original.columns)} Spalten")

# 2. Extrahierten Orders entfernen
df_cleaned, stats = remove_extracted_orders(
    df_original, 
    extracted_orders_file=extracted_orders_file,
    order_column='Order'
)
if 'error' not in stats:
    df_cleaned.to_csv('Ames_NoOutlier_'+data_file, index=False)
    print(f"\nBereinigter DataFrame gespeichert in: {'Ames_NoOutlier_'+data_file}")
    print("ZUSAMMENFASSUNG:")
    print(f"Original: {stats['original_count']} Datensätze")
    print(f"Entfernt: {stats['removed_count']} Datensätze")
    print(f"Verbleibend: {stats['remaining_count']} Datensätze")
    print(f"Reduktion: {stats['removed_count']/stats['original_count']*100:.1f}%")
    
    if stats['orders_not_found']:
        print(f"\nHinweis: {len(stats['orders_not_found'])} Orders wurden nicht gefunden.")



Verarbeite Daten aus Ames_Num_All_clean.csv
Originaler DataFrame: 2925 Zeilen, 186 Spalten
FEHLER: Datei Ames_GrossePreisunterschiede.csv nicht gefunden.
Stelle sicher, dass die Extraktionsfunktion zuerst ausgeführt wurde.


In [21]:
# Test, ob die Vorhersagen besser werden, wenn die Variablen durch Multiplikation mit label encoded Vororten besser werden. Abstände sollten dabei aufgeblasen werden.

df_mult_neighbor = df_num[['Order','SalePrice']].copy()

# Add the '1st Flr SF' column
df_mult_neighbor['Lot Area'] = df_num['Lot Area'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Gr Liv Area'] = df_num['Gr Liv Area'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Overall Qual'] = df_num['Overall Qual'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Exter Qual'] = df_num['Exter Qual'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Kitchen Qual'] = df_num['Kitchen Qual'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Garage Area'] = df_num['Garage Area'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['PorchSFTotal'] = df_num['PorchSFTotal'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['BathroomsTotal'] = df_num['BathroomsTotal'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['HouseAge'] = df_num['HouseAge'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['HouseRemodelAge'] = df_num['HouseRemodelAge'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Total Bsmt SF'] = df_num['Total Bsmt SF'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Railroad_Near'] = df_num['Railroad_Near'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Railroad_Adjacent'] = df_num['Railroad_Adjacent'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Condition_AdjLand'] = df_num['Sale Condition_AdjLand'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Condition_Alloca'] = df_num['Sale Condition_Alloca'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Condition_Normal'] = df_num['Sale Condition_Normal'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Condition_Partial'] = df_num['Sale Condition_Partial'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_COD'] = df_num['Sale Type_COD'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_CWD'] = df_num['Sale Type_CWD'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_Con'] = df_num['Sale Type_Con'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_ConLD'] = df_num['Sale Type_ConLD'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_ConLI'] = df_num['Sale Type_ConLI'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_ConLw'] = df_num['Sale Type_ConLw'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_New'] = df_num['Sale Type_New'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_Oth'] = df_num['Sale Type_Oth'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_VWD'] = df_num['Sale Type_VWD'] *  df_num['Neighbor_intsorted']
df_mult_neighbor['Sale Type_WD'] = df_num['Sale Type_WD'] *  df_num['Neighbor_intsorted']


df_mult_neighbor.to_csv('Ames_Small_MultNeighbor_sorted.csv', index=False)

In [22]:
df_mult_neighbor.info()

<class 'pandas.DataFrame'>
Index: 2925 entries, 0 to 2929
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order                   2925 non-null   int64  
 1   SalePrice               2925 non-null   int64  
 2   Lot Area                2925 non-null   int64  
 3   Gr Liv Area             2925 non-null   int64  
 4   Overall Qual            2925 non-null   int64  
 5   Exter Qual              2925 non-null   int64  
 6   Kitchen Qual            2925 non-null   int64  
 7   Garage Area             2925 non-null   float64
 8   PorchSFTotal            2925 non-null   int64  
 9   BathroomsTotal          2925 non-null   float64
 10  HouseAge                2925 non-null   int64  
 11  HouseRemodelAge         2925 non-null   int64  
 12  Total Bsmt SF           2925 non-null   float64
 13  Railroad_Near           2925 non-null   int64  
 14  Railroad_Adjacent       2925 non-null   int64  
 15  Sal